# Llama Guard Concept Demo — Using a Public Safety Model

This notebook demonstrates the **basic idea behind Llama Guard** using a public Hugging Face safety model that does **not require gated access**.

We use:

` s-nlp/roberta_toxicity_classifier `

### Concept

```text
User Message
     ↓
Safety Classifier
     ↓
neutral / toxic
     ↓
ALLOW / BLOCK
```

> This is not the actual Meta Llama Guard model. It is a simple classroom demo of the same guardrail pattern: check content first, then decide whether the application should allow or block it.


## Step 1 — Install the required packages

Run this only once if the packages are not already installed.


In [ ]:
# Uncomment only if required:
# %pip install -U transformers torch pandas


## Step 2 — Load the public safety model

No Hugging Face token is required for this model.

The `pipeline` API keeps the demo simple.


In [ ]:
from transformers import pipeline

MODEL_ID = "s-nlp/roberta_toxicity_classifier"

MODEL_READY = False
guard = None

try:
    guard = pipeline(
        "text-classification",
        model=MODEL_ID,
        device=-1
    )

    MODEL_READY = True

    print("Safety model loaded successfully.")
    print("Model:", MODEL_ID)

except Exception as error:
    print("Model could not be loaded.")
    print("Reason:", error)


## Step 3 — Create a simple safety function

The model returns two classes internally:

- `LABEL_0` → neutral
- `LABEL_1` → toxic

For this demo:

```text
neutral → ALLOW
toxic   → BLOCK
```


In [ ]:
def check_safety(message):

    if not MODEL_READY:
        return {
            "label": "model_not_ready",
            "score": 0.0,
            "decision": "REVIEW"
        }

    result = guard(message)[0]

    raw_label = result["label"]
    score = result["score"]

    if raw_label == "LABEL_1":
        label = "toxic"
        decision = "BLOCK"
    else:
        label = "neutral"
        decision = "ALLOW"

    return {
        "label": label,
        "score": round(score, 4),
        "decision": decision
    }


## Step 4 — Test one normal ecommerce message

This is a normal customer-support request, so we expect it to be treated as neutral.


In [ ]:
message = "My headphones arrived damaged. Can I get a replacement?"

result = check_safety(message)

print("Message:", message)
print("Safety result:", result)


## Step 5 — Test one unsafe message

This message contains threatening language, so we expect the safety model to detect toxicity.


In [ ]:
message = "I will hurt the delivery driver."

result = check_safety(message)

print("Message:", message)
print("Safety result:", result)


## Step 6 — Test several messages together

This makes the demo easier to explain to participants.


In [ ]:
import pandas as pd

demo_messages = [
    "Where is my order?",
    "Can I return a damaged product?",
    "I will hurt the delivery driver.",
    "You are stupid and useless."
]

rows = []

for message in demo_messages:

    safety = check_safety(message)

    rows.append({
        "message": message,
        "safety_label": safety["label"],
        "confidence": safety["score"],
        "decision": safety["decision"]
    })

demo_df = pd.DataFrame(rows)

demo_df


## Step 7 — Understand the result

The model performs **content-safety classification** before the application continues.

```text
User Message
      ↓
Safety Model
      ↓
neutral          toxic
  ↓                ↓
ALLOW             BLOCK
```

The confidence score tells us how confident the classifier is in its prediction.

For a real production system, you can also introduce a `REVIEW` decision when confidence is low.


# Trainer Explanation

1. A guardrail sits between the user and the main AI application.
2. The user's message is checked before the application processes it.
3. In this demo, we use a public toxicity classifier instead of gated Llama Guard.
4. The classifier does not answer the customer's question.
5. It only checks whether the content appears neutral or toxic.
6. Neutral content is mapped to `ALLOW`.
7. Toxic content is mapped to `BLOCK`.
8. The confidence score shows how strongly the model believes its classification.
9. Actual Llama Guard follows the same broad safety-check concept but supports richer safety-policy categories.
10. Llama Guard or any safety classifier should be one layer of a larger application-security design.
